# 08 - Data Quality Gate (job task)
Runs data quality checks across Bronze, Silver and Gold.
1. Every result is appended to `gold.data_quality_results` (audit trail per run)
2. If ANY check fails, the notebook raises an error -> the job fails -> email alert

**Job parameters:** `min_fact_rows` (threshold), `job_run_id` (passed from the job for traceability)

In [0]:
dbutils.widgets.text("min_fact_rows", "100000")
dbutils.widgets.text("job_run_id", "manual")

min_fact_rows = int(dbutils.widgets.get("min_fact_rows"))
job_run_id = dbutils.widgets.get("job_run_id")
C = "retail_lakehouse"

print(f"min_fact_rows = {min_fact_rows:,} | job_run_id = {job_run_id}")

In [0]:
import operator
from pyspark.sql import functions as F

OPS = {"==": operator.eq, ">=": operator.ge, "<=": operator.le}

# (check_name, layer, sql returning one number, operator, expected)
checks = [
    ("bronze_orders_fresh_26h", "bronze",
     f"SELECT CAST(MAX(_ingested_at) >= current_timestamp() - INTERVAL 26 HOURS AS INT) FROM {C}.bronze.orders_raw",
     "==", 1),
    ("silver_orders_no_null_ids", "silver",
     f"SELECT COUNT(*) FROM {C}.silver.orders WHERE order_id IS NULL",
     "==", 0),
    ("silver_orders_no_duplicates", "silver",
     f"SELECT COUNT(*) FROM (SELECT order_id FROM {C}.silver.orders GROUP BY order_id HAVING COUNT(*) > 1)",
     "==", 0),
    ("gold_fact_min_rows", "gold",
     f"SELECT COUNT(*) FROM {C}.gold.fact_orders",
     ">=", min_fact_rows),
    ("gold_fact_matches_silver_items", "gold",
     f"SELECT ABS((SELECT COUNT(*) FROM {C}.silver.order_items) - (SELECT COUNT(*) FROM {C}.gold.fact_orders))",
     "==", 0),
    ("gold_fact_missing_customer", "gold",
     f"SELECT COUNT(*) FROM {C}.gold.fact_orders WHERE customer_sk IS NULL",
     "==", 0),
    ("gold_fact_missing_product", "gold",
     f"SELECT COUNT(*) FROM {C}.gold.fact_orders WHERE product_sk IS NULL",
     "==", 0),
    ("gold_fact_no_duplicates", "gold",
     f"SELECT COUNT(*) FROM (SELECT order_id, order_item_id FROM {C}.gold.fact_orders GROUP BY 1, 2 HAVING COUNT(*) > 1)",
     "==", 0),
    ("gold_fact_no_negative_amounts", "gold",
     f"SELECT COUNT(*) FROM {C}.gold.fact_orders WHERE item_total < 0",
     "==", 0),
    ("gold_dim_customer_one_current", "gold",
     f"""SELECT COUNT(*) FROM (SELECT customer_id FROM {C}.gold.dim_customer GROUP BY customer_id
         HAVING SUM(CASE WHEN is_current THEN 1 ELSE 0 END) <> 1)""",
     "==", 0),
]

results = []
for name, layer, sql, op, expected in checks:
    actual = spark.sql(sql).first()[0]
    actual = int(actual) if actual is not None else -1
    passed = OPS[op](actual, expected)
    results.append((job_run_id, name, layer, actual, f"{op} {expected}", passed))
    print(f"{'✅' if passed else '❌'} {name:35} actual={actual:>10,}  expected {op} {expected:,}")

In [0]:
results_df = spark.createDataFrame(
    results,
    "job_run_id STRING, check_name STRING, layer STRING, actual_value BIGINT, rule STRING, passed BOOLEAN",
).withColumn("checked_at", F.current_timestamp())

results_df.write.mode("append").saveAsTable(f"{C}.gold.data_quality_results")
display(results_df)

failed = [r[1] for r in results if not r[5]]
if failed:
    raise Exception(f"❌ DATA QUALITY GATE FAILED: {len(failed)} check(s): {', '.join(failed)}")

print(f"✅ All {len(results)} data quality checks passed")